# CS4603 PA4 — Document Analyst

Development & testing notebook. Section headers match the tasks in `README.md`.
Fill in each cell, run everything top-to-bottom, and **keep all outputs visible** before submitting.
Record explanations and analysis answers in `STUDENT_ANALYSIS.md`.


## Part 0 — Setup & Corpus Ingestion
Env config + ingest `data/annual_report.pdf` into Databricks Vector Search (Task 0.3).


In [0]:
# TODO(0.1): load config / verify env vars
import os
from config import get_settings

# Task 0.1: configure and verify the non-secret Databricks resource names.
# Change these values only if you used different names during setup.

os.environ["SOURCE_TABLE"] = "cs4603.default.s27100380_analyst_chunks"
os.environ["VECTOR_SEARCH_ENDPOINT"] = "s27100380-vs-endpoint"
os.environ["VECTOR_SEARCH_INDEX"] = "cs4603.default.s27100380_analyst_index"
os.environ["EMBEDDINGS_ENDPOINT"] = "databricks-gte-large-en"

VOLUME_PATH = "/Volumes/cs4603/default/pa4/annual_report.pdf"

from rag.ingest import build_chunks_table

build_chunks_table(
    spark,
    volume_path="/Volumes/cs4603/default/pa4/annual_report.pdf",
    chunks_table="cs4603.default.s27100380_analyst_chunks",
)

for name in (
    "SOURCE_TABLE",
    "VECTOR_SEARCH_ENDPOINT",
    "VECTOR_SEARCH_INDEX",
    "EMBEDDINGS_ENDPOINT",
):
    assert os.environ.get(name), f"Missing {name}"
    print(f"{name}={os.environ[name]}")

display(dbutils.fs.ls("/Volumes/cs4603/default/pa4/"))


In [0]:
%pip install databricks-ai-search

In [0]:
dbutils.library.restartPython()

In [0]:
# TODO(0.3): ingest corpus -> Delta table -> Vector Search index; wait until READY
# from rag.ingest import ingest
# ingest(spark, volume_path='/Volumes/main/default/pa4/annual_report.pdf')
# Task 0.3: parse, chunk, create/sync the index, wait for READY, and test it.
from rag.ingest import ingest

index = ingest(spark, volume_path=VOLUME_PATH)

display(
    spark.sql(
        f"""
        SELECT chunk_id, source, page, chunk_to_retrieve, chunk_to_embed
        FROM {os.environ['SOURCE_TABLE']}
        ORDER BY page
        """
    )
)

index.describe()



In [0]:
from rag.ingest import verify_index
print(index.describe())
verify_index(
    index,
    query="What was the net income in 2023?",
)

In [0]:
display(
    spark.sql("""
        SELECT chunk_id, source, page, chunk_to_retrieve
        FROM cs4603.default.s27100380_analyst_chunks
        ORDER BY page
    """)
)

## Part 1 — Build the Document Analyst graph
Nodes: planner (1.2), supervisor (1.3), RAG agent (1.4), MCP tools (1.5), synthesizer (1.6), full graph (1.7).


In [0]:
# TODO(1.7): build the compiled graph
# from agent.graph import build_graph
# graph = build_graph()



In [0]:
# TODO(1.7): visualize the compiled graph
# from IPython.display import Image
# Image(graph.get_graph().draw_mermaid_png())



### Test the graph


In [0]:
# Retrieval-only query
# graph.invoke({'messages':[{'role':'user','content':'What was the net income in 2023?'}]})



In [0]:
# Computation-only query
# graph.invoke({'messages':[{'role':'user','content':'What is 15% of 2.4 billion?'}]})



In [0]:
# Combined query — show the full step-by-step execution trace
# graph.invoke({'messages':[{'role':'user','content':'What was the revenue in 2023, and what would a 10% increase look like?'}]})



### Required — offline smoke test
Runs the graph with a mocked LLM (no Databricks calls). Same test Bonus A automates.


In [0]:
!python -m pytest tests/test_smoke.py -q


## Part 2 — Deployment
Package as an MLflow models-from-code model, register in Unity Catalog, create the serving endpoint (Tasks 2.1–2.4).
Reference: `databricks_deployment_v1/deployment.ipynb`.


In [0]:
# TODO(2.1): sanity-check the model definition imports cleanly
!python -c "import deployment.agent_model"



In [0]:
# TODO(2.2): log + register the model version in Unity Catalog



In [0]:
# TODO(2.3): create/update the serving endpoint; wait for READY; print the URL



### Test the deployed endpoint (Task 2.4)


In [0]:
# curl the endpoint and show the raw response



In [0]:
# Response shape depends on how you logged the model (see README Task 2.4 / GUIDE §7).
#
# Path A — raw LangGraph state (mlflow.langchain.log_model, v1 style):
# import requests
# url = f'{DATABRICKS_HOST}/serving-endpoints/<your-endpoint-name>/invocations'
# r = requests.post(url, headers={'Authorization': f'Bearer {DATABRICKS_TOKEN}'},
#                   json={'messages':[{'role':'user','content':'What was the net income in 2023?'}]})
# print(r.json()[0]['messages'][-1]['content'])
#
# Path B — OpenAI ChatCompletion (mlflow.pyfunc.ChatModel / ChatAgent, v2 style):
# import openai
# client = openai.OpenAI(api_key=DATABRICKS_TOKEN, base_url=f'{DATABRICKS_HOST}/serving-endpoints')
# resp = client.chat.completions.create(model='<your-endpoint-name>',
#     messages=[{'role':'user','content':'What was the net income in 2023?'}])
# print(resp.choices[0].message.content)



## Part 3 — Client SDK demo
Instantiate `DocumentAnalystClient`, health-check, ask, stream, and show timeout/retry handling (Task 3.2).


In [0]:
# from client.sdk import DocumentAnalystClient
# c = DocumentAnalystClient(...)
# assert c.health_check() is True
# print(c.ask('What was the net income in 2023?'))



In [0]:
# ask_streaming demo
# for chunk in c.ask_streaming('Summarize FY2023 revenue.'): print(chunk, end='')



In [0]:
# Simulate timeout (timeout=0.001) and endpoint-unavailable retry behavior

